## **GET METADATA FROM DATAHUB AND CONVERT INTO JSONL**

In [2]:
# Smart RAG Ingest: JSONL -> Chroma (Ollama embeddings) with time-aware upsert
# Shows LangGraph inline in a Jupyter notebook (Mermaid only)
# FINAL: SQL-aware column cards, top-placed ALLOWED IDENTIFIERS (table-narrowing),
# stricter prompt, alias->canonical fixer, JOIN pruning, and whitelist validation+regeneration.

import os
import re
import json
import glob
import hashlib
from pathlib import Path
from typing import TypedDict, List, Dict, Any, Optional, Iterable
from datetime import datetime
from collections import defaultdict

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

from langgraph.graph import StateGraph, END
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

from IPython.display import HTML, display  # for notebook graph rendering

# ----------------- Config (env with SAFE DEFAULTS) -----------------
# Accepts single file, directory, or glob (e.g., './datahub_metadata/*.jsonl')
JSON_INPUT = os.getenv("EXPORT_JSONL_OUTPUT", "./datahub_metadata/*.jsonl").strip()
CHROMA_DIR = os.getenv("CHROMA_DIR", "./chroma_db_test")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "test_collection")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_BASE_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
CHAT_MODEL = os.getenv("CHAT_MODEL", "qwen2.5:3b")
TOP_K = int(os.getenv("TOP_K", "5"))

# Retrieval probe to bias toward measures, dates, and common dims
RETRIEVAL_PROBE = (
    " measure measures metric metrics sum total amount price revenue sales gross net "
    "quantity qty count average avg mean distinct group by bucket date time timestamp "
    "created updated order_date month week quarter year category region customer status id"
)

# ----------------- LangGraph state -----------------
class RAGState(TypedDict):
    jsonl_paths: List[str]
    question: str
    added: int
    updated: int
    skipped: int
    context: str
    answer: str

# ----------------- Helpers -----------------
def resolve_jsonl_paths(spec: str) -> List[str]:
    paths: List[str] = []
    if not spec:
        return paths
    p = Path(spec)
    if any(ch in spec for ch in "*?[]"):
        paths.extend(glob.glob(spec))
    elif p.is_dir():
        paths.extend(str(x) for x in p.glob("*.jsonl"))
    elif p.is_file():
        paths.append(str(p))
    else:
        paths.extend(glob.glob(spec))
    out, seen = [], set()
    for x in paths:
        xp = str(Path(x))
        if xp.lower().endswith(".jsonl") and xp not in seen:
            seen.add(xp)
            out.append(xp)
    return out

def build_vector_store() -> Chroma:
    embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)
    return Chroma(
        persist_directory=CHROMA_DIR,
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
    )

def read_jsonl(path: str) -> Iterable[tuple[str, Optional[Dict[str, Any]]]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
            except Exception:
                obj = None
            yield s, obj

def _safe_iso_to_dt(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return None

def _stable_id(meta: Dict[str, Any], source_name: str) -> str:
    key = f"{meta.get('database','')}|{meta.get('table','')}|{meta.get('column','')}|{source_name}"
    return hashlib.sha1(key.encode("utf-8")).hexdigest()

def show_graph_mermaid_only(app) -> None:
    G = app.get_graph()
    mermaid_src = G.draw_mermaid()
    html = f"""
<div class="mermaid">
{mermaid_src}
</div>
<script>
(function () {{
  function renderMermaid() {{
    if (window.mermaid) {{
      mermaid.initialize({{ startOnLoad: true, securityLevel: 'loose' }});
      mermaid.init();
    }}
  }}
  if (!window.mermaid) {{
    var s = document.createElement('script');
    s.src = 'https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js';
    s.onload = renderMermaid;
    document.body.appendChild(s);
  }} else {{
    renderMermaid();
  }}
}})();
</script>
"""
    display(HTML(html))

# --- SQL-aware "column card" for better embeddings ---
def _column_card(meta: Dict[str, Any]) -> str:
    db = meta.get("database", "") or meta.get("db", "")
    sch = meta.get("schema", "") or meta.get("schemaName", "")
    tbl = meta.get("table", "") or meta.get("tableName", "")
    col = meta.get("column", "") or meta.get("columnName", "")
    dtype = meta.get("dtype") or meta.get("dataType") or meta.get("type", "")
    desc = meta.get("description") or meta.get("brief", "") or ""
    aliases = meta.get("aliases") or meta.get("synonyms") or meta.get("aka") or []
    if isinstance(aliases, str):
        aliases = [aliases]
    sample_vals = meta.get("sample_values") or meta.get("samples") or meta.get("examples") or []
    if isinstance(sample_vals, dict):
        sample_vals = list(sample_vals.values())
    sample_txt = ", ".join(map(lambda x: str(x)[:40], sample_vals[:5]))

    fqn = ".".join([p for p in [db, sch, tbl, col] if p])
    return (
        f"TABLE: {tbl}\n"
        f"COLUMN: {col}\n"
        f"TYPE: {dtype}\n"
        f"ALIASES: {', '.join(map(str, aliases))}\n"
        f"DESC: {desc}\n"
        f"SAMPLE: {sample_txt}\n"
        f"FQN: {fqn}"
    ).strip()

# ----------------- Nodes -----------------
def ingest_node(state: RAGState) -> RAGState:
    paths = state["jsonl_paths"]
    vs = build_vector_store()

    total_in_collection = vs._collection.count()
    first_run = (total_in_collection == 0)
    if first_run:
        print("🚀 First run detected (empty collection) — embedding all documents.")

    added = 0
    updated = 0
    skipped = 0

    for path in paths:
        p = Path(path)
        if not p.exists():
            print(f"⚠️ JSONL not found: {path}")
            continue

        docs: List[Document] = []
        ids: List[str] = []
        metas: List[Dict[str, Any]] = []

        for raw, obj in read_jsonl(path):
            meta: Dict[str, Any] = obj if isinstance(obj, dict) else {"raw": raw}
            meta["_source"] = p.name
            metas.append(meta)
            ids.append(_stable_id(meta, p.name))
            # Use SQL-aware card instead of raw JSON line
            docs.append(Document(page_content=_column_card(meta), metadata=meta))

        if not docs:
            continue

        if first_run:
            vs.add_documents(docs, ids=ids)
            added += len(docs)
            print(f"✅ Embedded {len(docs)} docs from {path}")
            continue

        BATCH = 1000
        existing_meta_by_id: Dict[str, Dict[str, Any]] = {}
        for i in range(0, len(ids), BATCH):
            chunk_ids = ids[i:i + BATCH]
            got = vs._collection.get(ids=chunk_ids, include=["metadatas"])
            got_ids = got.get("ids") or []
            got_metas = got.get("metadatas") or []
            for doc_id, md in zip(got_ids, got_metas):
                existing_meta_by_id[doc_id] = md or {}

        to_upsert_docs: List[Document] = []
        to_upsert_ids: List[str] = []

        for doc, doc_id, meta in zip(docs, ids, metas):
            incoming_li = meta.get("last_ingested")
            existing = existing_meta_by_id.get(doc_id)

            if not existing:
                to_upsert_docs.append(doc)
                to_upsert_ids.append(doc_id)
                continue

            stored_li = existing.get("last_ingested")
            should_update = (incoming_li != stored_li)

            in_dt = _safe_iso_to_dt(incoming_li)
            st_dt = _safe_iso_to_dt(stored_li)
            if in_dt and st_dt:
                should_update = in_dt > st_dt

            if should_update:
                to_upsert_docs.append(doc)
                to_upsert_ids.append(doc_id)
            else:
                skipped += 1

        if to_upsert_docs:
            vs.delete(ids=to_upsert_ids)
            vs.add_documents(to_upsert_docs, ids=to_upsert_ids)
            updated += len(to_upsert_docs)
            print(f"🔁 Upserted {len(to_upsert_docs)} changed docs from {path}")

    try:
        vs.persist()
    except Exception:
        pass

    state["added"] = added
    state["updated"] = updated
    state["skipped"] = skipped
    return state

def retrieve_node(state: RAGState) -> RAGState:
    vs = build_vector_store()
    retriever = vs.as_retriever(search_kwargs={"k": max(3, TOP_K)})
    q = (state["question"] or "").strip()
    hybrid_query = (q + " " + RETRIEVAL_PROBE).strip()
    docs = retriever.invoke(hybrid_query)

    # Build alias map + per-table allowed
    allowed = defaultdict(list)      # table -> [cols]
    alias_map = defaultdict(dict)    # table -> alias(lower) -> canonical
    for d in (docs or []):
        md = d.metadata or {}
        t = (md.get("table") or md.get("tableName") or "").strip()
        c = (md.get("column") or md.get("columnName") or "").strip()
        if t and c:
            allowed[t].append(c)
            aliases = md.get("aliases") or md.get("synonyms") or []
            if isinstance(aliases, str):
                aliases = [aliases]
            for al in aliases:
                if al:
                    alias_map[t][str(al).lower()] = c

    # Heuristic: narrow to likely table(s) based on question tokens
    q_l = q.lower()
    likely_tables = set()
    if any(w in q_l for w in ["customer", "customers", "customer name", "customer country"]):
        likely_tables.add("customers")
    if any(w in q_l for w in ["order", "orders", "order date", "order status"]):
        likely_tables.add("orders")
    if any(w in q_l for w in ["product", "products", "productline", "product line"]):
        likely_tables.add("products")

    if likely_tables:
        allowed = {t: cols for t, cols in allowed.items() if t.lower() in likely_tables}
    else:
        allowed = dict(allowed)  # keep all if nothing matched

    # Whitelist (top) + cards (bottom)
    allowed_lines = []
    for t, cols in allowed.items():
        uniq = sorted({*cols})
        allowed_lines.append(f"- {t}: {', '.join(uniq)}")
    allowed_block = "### ALLOWED IDENTIFIERS\n" + "\n".join(allowed_lines) if allowed_lines else "### ALLOWED IDENTIFIERS\n- (none)"

    cards_text = "\n".join(d.page_content for d in (docs or []))
    context = (allowed_block + "\n\n" + cards_text).strip()

    # Stash alias map (JSON) for the post-fix step
    context += f"\n\n### _INTERNAL_HINT\n{json.dumps({'alias_map': alias_map}, ensure_ascii=False)}"

    state["context"] = context
    return state

# ---------- SQL post-processing helpers ----------
_AGG_WORDS = re.compile(r"\b(sum|avg|count|min|max)\s*\(", re.I)
JOIN_KW = r"(?:inner\s+join|left\s+join|right\s+join|full\s+join|join)"  # includes plain JOIN

def _normalize_cols(s: str) -> List[str]:
    parts = [p.strip() for p in s.split(",")]
    out = []
    for p in parts:
        m = re.split(r"\s+AS\s+", p, flags=re.I)
        left = m[0]
        out.append(left.strip())
    return out

def _has_join(sql: str) -> bool:
    return re.search(rf"\b{JOIN_KW}\b", sql, flags=re.I) is not None

def _prefer_distinct_over_group_by(sql: str) -> str:
    m = re.search(r"^\s*select\s+(.*?)\s+from\s+(.*?)(\s+group\s+by\s+(.*?))(?:\s+order\s+by\s+.*)?;", sql, flags=re.I | re.S)
    if not m:
        return sql
    select_cols = _normalize_cols(m.group(1))
    group_by_cols = _normalize_cols(m.group(4))
    if _AGG_WORDS.search(sql):
        return sql
    if select_cols and group_by_cols and set(map(str.lower, select_cols)) == set(map(str.lower, group_by_cols)):
        sql2 = re.sub(r"^\s*select\s+", "SELECT DISTINCT ", sql, flags=re.I)
        sql2 = re.sub(r"\s+group\s+by\s+.*?(?=(\s+order\s+by\s+.*)?;)", "", sql2, flags=re.I | re.S)
        return sql2
    return sql

# Recognize anchor even without alias, and with backticks/FQNs
def _find_anchor_alias(sql: str) -> Optional[str]:
    m = re.search(
        r"""\bfrom\s+
            `?(?:([\w]+)\.)?([\w]+)`?
            (?:\s+as\s+`?([\w]+)`?|\s+`?([\w]+)`?)?
        """,
        sql, flags=re.I | re.X
    )
    if not m:
        return None
    _schema, table, alias1, alias2 = m.group(1), m.group(2), m.group(3), m.group(4)
    return (alias1 or alias2 or table)

def _list_join_aliases(sql: str) -> List[str]:
    return [m.group(2).strip('`"') for m in re.finditer(rf"\b{JOIN_KW}\s+[`\"\w\.]+\s+([`\"\w]+)\s+on\b", sql, flags=re.I)]

def _remove_redundant_joins(sql: str) -> str:
    joins = list(re.finditer(rf"\b({JOIN_KW})\s+([`\"\w\.]+)\s+([`\"\w]+)\s+on\s+([^;]+?)(?=\b({JOIN_KW}|where|group|order|limit|;)\b)", sql, flags=re.I | re.S))
    if not joins:
        return sql

    def alias_used_outside_on(alias: str, sql_text: str, on_span: tuple[int,int]) -> bool:
        alias_pat = re.compile(rf"\b{re.escape(alias)}\s*\.", re.I)
        before = sql_text[:on_span[0]]
        after  = sql_text[on_span[1]:]
        return bool(alias_pat.search(before) or alias_pat.search(after))

    new_sql = sql
    offset = 0
    for j in joins:
        alias = j.group(3).strip("`\"")
        on_start = j.start(4) + offset
        on_end   = j.end(4) + offset
        block_span = (j.start() + offset, j.end() + offset)
        used = alias_used_outside_on(alias, new_sql, (on_start, on_end))
        if not used:
            new_sql = new_sql[:block_span[0]] + new_sql[block_span[1]:]
            offset -= (block_span[1] - block_span[0])
    return new_sql

def _strip_on_clauses(sql: str) -> str:
    return re.sub(rf"\bon\s+([^;]+?)(?=\b({JOIN_KW}|where|group|order|limit|;)\b)", " ON ", sql, flags=re.I | re.S)

def _aliases_used_outside_on(sql: str) -> set:
    stripped = _strip_on_clauses(sql)
    return set(m.group(1).strip('`"') for m in re.finditer(r"\b([`\"\w]+)\s*\.", stripped))

def _pushdown_filters_through_equi_join(sql: str) -> str:
    anchor = _find_anchor_alias(sql)
    if not anchor:
        return sql

    pattern = re.compile(rf"\b({JOIN_KW})\s+([`\"\w\.]+)\s+([`\"\w]+)\s+on\s+([^;]+?)(?=\b({JOIN_KW}|where|group|order|limit|;)\b)", re.I | re.S)
    alias_col_map: Dict[str, str] = {}
    joins = list(pattern.finditer(sql))
    for j in joins:
        alias = j.group(3).strip('`"')
        on_text = j.group(4)
        for m in re.finditer(r"\b([`\"\w]+)\.([`\"\w]+)\s*=\s*([`\"\w]+)\.([`\"\w]+)\b", on_text):
            a1, c1, a2, c2 = (m.group(1).strip('`"'), m.group(2).strip('`"'), m.group(3).strip('`"'), m.group(4).strip('`"'))
            if a1 == anchor and a2 == alias:
                alias_col_map[f"{alias}.{c2}"] = f"{anchor}.{c1}"
            elif a2 == anchor and a1 == alias:
                alias_col_map[f"{alias}.{c1}"] = f"{anchor}.{c2}"

    if not alias_col_map:
        return sql

    new_sql = sql
    for j in joins:
        on_span = (j.start(4), j.end(4))
        before = new_sql[:on_span[0]]
        onpart = new_sql[on_span[0]:on_span[1]]
        after  = new_sql[on_span[1]:]
        for alias_col, anchor_col in alias_col_map.items():
            alias_name, col_name = alias_col.split(".")
            alias_regex = re.compile(rf"\b{re.escape(alias_name)}\s*\.\s*{re.escape(col_name)}\b", re.I)
            before = alias_regex.sub(anchor_col, before)
            after  = alias_regex.sub(anchor_col, after)
        new_sql = before + onpart + after

    new_sql = _remove_redundant_joins(new_sql)
    return new_sql

def _drop_all_joins_if_only_anchor_used(sql: str) -> str:
    anchor = _find_anchor_alias(sql)
    if not anchor:
        return sql
    used = _aliases_used_outside_on(sql)
    if used.issubset({anchor}):
        return re.sub(rf"\s+{JOIN_KW}\s+[`\"\w\.]+\s+[`\"\w]+\s+on\s+[^;]+?(?=\b({JOIN_KW}|where|group|order|limit|;)\b)", "", sql, flags=re.I | re.S)
    return sql

# If all references outside ON belong to a single qualifier (table/alias), drop all JOINs.
def _strip_joins_when_only_single_qualifier_used(sql: str) -> str:
    quals = _aliases_used_outside_on(sql)
    if len(quals) == 1:
        return re.sub(
            rf"\s+{JOIN_KW}\s+[`\"\w\.]+\s+[`\"\w]+\s+on\s+[^;]+?(?=\b({JOIN_KW}|where|group|order|limit|;)\b)",
            "", sql, flags=re.I | re.S
        )
    return sql

# NEW: drop DISTINCT when it's provably unnecessary (no joins/agg/group and anchor has id-like col)
_IDLIKE = re.compile(r".*\b(id|.*_id|.*id$|.*number$|.*_number)\b", re.I)

def _select_has_anchor_id_like(sql: str, anchor: Optional[str]) -> bool:
    if not anchor:
        return False
    m = re.search(r"^\s*select\s+(.*?)\s+from\s+", sql, flags=re.I | re.S)
    if not m:
        return False
    cols = _normalize_cols(m.group(1))
    for col in cols:
        m2 = re.match(r"(?:(?:`?)([\w]+)(?:`?)\s*\.\s*(?:`?)([\w]+)(?:`?))", col)
        if not m2:
            continue
        alias, cname = m2.group(1), m2.group(2)
        if alias.strip('`"').lower() == anchor.lower() and _IDLIKE.match(cname):
            return True
    return False

def _drop_unnecessary_distinct(sql: str) -> str:
    if "distinct" not in sql.lower():
        return sql
    if _has_join(sql) or " group by " in sql.lower() or _AGG_WORDS.search(sql):
        return sql
    anchor = _find_anchor_alias(sql)
    if _select_has_anchor_id_like(sql, anchor):
        return re.sub(r"^\s*select\s+distinct\s+", "SELECT ", sql, flags=re.I)
    return sql

# Normalize backticks around FQNs like `table.column` -> `table`.`column`
def _normalize_fqn_backticks(sql: str) -> str:
    return re.sub(
        r"`([A-Za-z_][A-Za-z0-9_]*)\.([A-Za-z_][A-Za-z0-9_]*)`",
        r"`\1`.`\2`",
        sql
    )

def _postprocess_sql(sql: str) -> str:
    if not sql:
        return sql
    s = sql.strip()
    if not s.endswith(";"):
        s += ";"
    s = _normalize_fqn_backticks(s)
    s = _pushdown_filters_through_equi_join(s)
    s = _drop_all_joins_if_only_anchor_used(s)
    s = _strip_joins_when_only_single_qualifier_used(s)
    s = _remove_redundant_joins(s)
    s = _prefer_distinct_over_group_by(s)
    s = _drop_unnecessary_distinct(s)
    return s

# --- alias-map extraction (for repairs) ---
def _extract_internal_hint(ctx: str) -> Dict[str, Dict[str, str]]:
    m = re.search(r"### _INTERNAL_HINT\s+({.*})\s*$", ctx, flags=re.S)
    if not m:
        return {}
    try:
        obj = json.loads(m.group(1))
        return obj.get("alias_map") or {}
    except Exception:
        return {}

# ---------- Whitelist parsing + validation ----------
_ALNUM = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")

def _parse_allowed_identifiers(ctx: str) -> Dict[str, set]:
    """Parse the '### ALLOWED IDENTIFIERS' block into {table: {cols}}."""
    tables: Dict[str, set] = {}
    m = re.search(r"### ALLOWED IDENTIFIERS(.*?)(?:\n### |\Z)", ctx, flags=re.S)
    if not m:
        return tables
    block = m.group(1)
    for line in block.splitlines():
        line = line.strip()
        if not line.startswith("- "):
            continue
        # "- customers: customerName, country, city"
        parts = line[2:].split(":", 1)
        if len(parts) != 2:
            continue
        table = parts[0].strip().strip("`")
        cols = [c.strip().strip("`") for c in parts[1].split(",") if c.strip()]
        if table:
            tables.setdefault(table, set()).update(cols)
    return tables

def _identify_used_identifiers(sql: str) -> Dict[Optional[str], set]:
    """Return {qualifier -> {columns}} for qualified refs, and {None}->{bare_cols} for bare."""
    qualified = re.findall(r"`?([A-Za-z_][A-Za-z0-9_]*)`?\s*\.\s*`?([A-Za-z_][A-Za-z0-9_]*)`?", sql)
    out: Dict[Optional[str], set] = {}
    for t, c in qualified:
        out.setdefault(t, set()).add(c)
    # bare identifiers (not keywords)
    keywords = {"select","from","where","group","by","order","limit","join","left","right","inner","on","and","or","not","as","distinct","count","sum","avg","min","max","having","asc","desc","like","in","between","is","null","case","when","then","else","end"}
    for tok in re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\b", sql):
        if tok.lower() in keywords:
            continue
        out.setdefault(None, set()).add(tok)
    return out

def _validate_identifiers(sql: str, ctx: str) -> List[str]:
    """Return a list of violation strings if any identifier is not in the whitelist."""
    allowed = _parse_allowed_identifiers(ctx)  # {table: {cols}}
    used = _identify_used_identifiers(sql)
    errs: List[str] = []
    # qualified checks
    for tbl, cols in used.items():
        if tbl is None:
            continue
        if tbl not in allowed:
            errs.append(f"table `{tbl}` not in ALLOWED")
            continue
        bad = [c for c in cols if c not in allowed[tbl]]
        if bad:
            errs.append(f"{tbl} -> invalid columns: {', '.join(bad)}")
    return errs

# ---------- Repairs using aliases ----------
def _repair_with_aliases(sql: str, ctx: str) -> str:
    """Replace bare aliases with canonical names (unambiguous only)."""
    alias_map_per_table = _extract_internal_hint(ctx)
    if not alias_map_per_table:
        return sql
    # Build global alias->canonical if unique
    global_map: Dict[str, Optional[str]] = {}
    for _t, amap in alias_map_per_table.items():
        for al, canon in amap.items():
            al = str(al).lower()
            if al not in global_map:
                global_map[al] = canon
            elif global_map[al] != canon:
                global_map[al] = None  # ambiguous
    keywords = {"select","from","where","group","by","order","limit","join","left","right","inner","on","and","or","not","as","distinct","count","sum","avg","min","max","having","asc","desc","like","in","between","is","null","case","when","then","else","end"}
    pattern = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\b")
    def smart_repl(m):
        w = m.group(0)
        lw = w.lower()
        if lw in keywords:
            return w
        canon = global_map.get(lw)
        return canon if canon and _ALNUM.match(canon) else w
    return pattern.sub(smart_repl, sql)

# ---------- Prompt ----------
def _make_generation_prompt(ctx: str, question: str) -> str:
    return f"""
You are a senior SQL assistant for MySQL 8.0.

Return a runnable query using ONLY tables/columns listed in the ALLOWED IDENTIFIERS section of Context.
If a needed column name appears as an alias in Context, use its canonical column name.

########################
# STRICT OUTPUT FORMAT #
########################
{{ "sql": "<final MySQL query>", "notes": "<brief rationale; for EACH JOIN, name the exact selected/grouped/filtered column it contributes>" }}
# No extra keys. No Markdown. End SQL with a semicolon.
# If the final SQL has NO JOINs, set "notes" exactly to: "No joins used."

##################
# SQL CONSTRAINT #
##################
- MySQL 8.0; backticks only when needed; ANSI joins.

########################################
# ANCHOR TABLE & MINIMAL-JOIN POLICY   #
########################################
- Pick as ANCHOR the table containing the most required columns.
- If the ANCHOR already has a needed field by name/description, USE IT. Do NOT JOIN another table for a duplicate.
- Only JOIN if you SELECT, GROUP BY, or WHERE on a column that is NOT in the ANCHOR.
- If an equi-join ties ANCHOR.col = DIM.col and WHERE only constrains DIM.col with a literal, PUSH the filter to ANCHOR.col and REMOVE the JOIN unless DIM also contributes a selected/grouped column.
- After drafting, AUDIT each JOIN: if you cannot state which output/filter column comes from that table, REMOVE it.

########################################
# DISTINCT & GROUP BY
########################################
- Avoid DISTINCT unless: (a) explicitly asked, or (b) a JOIN creates duplicates in output columns.
- If dedup without aggregation is intended, prefer SELECT DISTINCT over GROUP BY; otherwise avoid both.

########################################
# DATE BUCKETING (MySQL-safe)
########################################
- DATE(dt) AS day; DATE_FORMAT(dt,'%Y-%m') AS month; CONCAT(YEAR(dt),'-Q',QUARTER(dt)) AS quarter; YEAR(dt) AS year.
- Always GROUP BY any selected bucket/dimension.

#################################
# FILTERS & TYPES
#################################
- Do not invent date ranges. Cast textual dates with STR_TO_DATE if needed.
- Only add WHERE conditions clearly implied by the Question.

################
# CONTEXT
################
{ctx}

Question:
{question}
""".strip()

# ---------- Notes sanitizer (NEW) ----------
def _sanitize_notes(sql: str, notes: str) -> str:
    """Ensure notes reflect the actual SQL. If no JOIN is present, avoid any join claims."""
    if not notes:
        return notes
    has_join = _has_join(sql or "")
    clean = notes.strip()
    if not has_join:
        # If the text talks about joins but SQL has none, standardize it.
        if re.search(r"\bjoin\b", clean, flags=re.I):
            return "No joins used."
        return "No joins used."
    return clean

# ---------- Generation (with validation & regeneration) ----------
def generate_node(state: RAGState) -> RAGState:
    llm = ChatOllama(model=CHAT_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.0)
    ctx = state.get("context", "")
    question = state.get("question", "")
    prompt = _make_generation_prompt(ctx, question)

    def run_once(prompt_text: str):
        resp = llm.invoke(prompt_text)
        content = getattr(resp, "content", None) or str(resp)
        try:
            obj = json.loads(content)
            raw_sql = (obj.get("sql") or "").strip()
            notes = (obj.get("notes") or "").strip()
            return raw_sql, notes
        except Exception:
            return content, ""

    raw_sql, notes = run_once(prompt)

    # Pass 1: post-process + alias repair + post-process again
    sql = _postprocess_sql(raw_sql) if raw_sql else raw_sql
    if sql:
        sql = _repair_with_aliases(sql, ctx)
        sql = _postprocess_sql(sql)

    # Validate identifiers against whitelist
    violations = _validate_identifiers(sql or "", ctx)
    if violations:
        # One more attempt: force the model to rewrite strictly with the whitelist
        force_msg = (
            "The previous SQL used identifiers not present in ALLOWED IDENTIFIERS:\n- "
            + "\n- ".join(violations)
            + "\nRewrite the SQL using ONLY identifiers listed in ALLOWED IDENTIFIERS. "
              "If a synonym appears, choose the canonical column."
        )
        strict_prompt = _make_generation_prompt(ctx + "\n\n" + force_msg, question)
        raw_sql2, notes2 = run_once(strict_prompt)
        sql2 = _postprocess_sql(raw_sql2) if raw_sql2 else raw_sql2
        if sql2:
            sql2 = _repair_with_aliases(sql2, ctx)
            sql2 = _postprocess_sql(sql2)
        # Prefer result with fewer violations
        v1 = len(violations)
        v2 = len(_validate_identifiers(sql2 or "", ctx))
        if v2 <= v1 and sql2:
            sql, notes = sql2, (notes2 or notes)

    # ---> APPLY NOTES SANITIZER (NEW)
    notes = _sanitize_notes(sql or "", notes or "")

    state["answer"] = (sql + ("\n\n-- notes: " + notes if notes else "")) if sql else "(no answer)"
    return state

# ----------------- Build graph -----------------
def build_app():
    g = StateGraph(RAGState)
    g.add_node("ingest", ingest_node)
    g.add_node("retrieve", retrieve_node)
    g.add_node("generate", generate_node)
    g.set_entry_point("ingest")
    g.add_edge("ingest", "retrieve")
    g.add_edge("retrieve", "generate")
    g.add_edge("generate", END)
    return g.compile()

# ----------------- Run once with REPL loop -----------------
def main():
    paths = resolve_jsonl_paths(JSON_INPUT)
    if not paths:
        raise RuntimeError(
            "No JSONL inputs found.\n"
            "Set COLUMNS_OUTPUT_DIR to one of:\n"
            "  • a single file: ./exports/classicmodels.jsonl\n"
            "  • a directory:   ./datahub_metadata\n"
            "  • a glob:        ./datahub_metadata/*.jsonl\n"
        )

    print("📄 JSONL inputs:")
    for p in paths:
        print(" -", p)

    first_q = input("Enter a test question (or press Enter to skip): ").strip()
    if not first_q:
        first_q = "Which columns describe customer identity?"

    app = build_app()
    show_graph_mermaid_only(app)

    init: RAGState = {
        "jsonl_paths": paths,
        "question": first_q,
        "added": 0,
        "updated": 0,
        "skipped": 0,
        "context": "",
        "answer": "",
    }
    out = app.invoke(init)

    vs = build_vector_store()
    total = vs._collection.count()

    print("\n=== RAG Ingest Report ===")
    print(f"Added (first run): {out.get('added', 0)}")
    print(f"Updated (upserted): {out.get('updated', 0)}")
    print(f"Skipped (unchanged): {out.get('skipped', 0)}")
    print(f"Total docs now in collection: {total}")
    print(f"Chroma path: {Path(CHROMA_DIR).resolve()} | Collection: {COLLECTION_NAME}")

    print("\n--- Answer ---\n")
    print(out.get("answer", "(no answer)"))

    print("\n🔁 Ask me more questions! (type 'exit' to quit)")
    try:
        while True:
            q = input("\nQuestion> ").strip()
            if not q or q.lower() in {"exit", "quit", "q"}:
                print("Bye.")
                break
            state: RAGState = {
                "jsonl_paths": paths,
                "question": q,
                "added": 0,
                "updated": 0,
                "skipped": 0,
                "context": "",
                "answer": "",
            }
            try:
                state = retrieve_node(state)
                state = generate_node(state)
                print("\n--- Answer ---\n")
                print(state.get("answer", "(no answer)"))
            except Exception as e:
                print("\n[Error] Failed to answer:", e)
    except KeyboardInterrupt:
        print("\nInterrupted. Bye.")

if __name__ == "__main__":
    main()


📄 JSONL inputs:
 - metadata_jsonl/classicmodels.jsonl


Enter a test question (or press Enter to skip):  List all customer names (alphabetical).



=== RAG Ingest Report ===
Added (first run): 0
Updated (upserted): 0
Skipped (unchanged): 59
Total docs now in collection: 59
Chroma path: /app/chroma_db | Collection: classicmodels

--- Answer ---

SELECT c.customerName FROM customers c ORDER BY c.customerName;

-- notes: No joins used.

🔁 Ask me more questions! (type 'exit' to quit)



Question>  Show distinct customer countries.



--- Answer ---

SELECT DISTINCT customers.country FROM classicmodels.customers;

-- notes: No joins used.



Question>  How many orders exist?



--- Answer ---

SELECT COUNT(*) FROM classicmodels.orders;

-- notes: No joins used.



Question>  exit


Bye.
